# RDKV on Kaggle (GPU T4 x2): Team 6, v4
Uses the team harness (`harness/run_longbench.py`) from github.com/samruddhichitnis02/298a-kv-cache-compression.

**Setup:** Accelerator **GPU T4 x2**, Internet **On**. Your Hugging Face token is a secret (`HF_TOKEN` or `hugging face`). Input: **rdkv_kaggle_v4.zip**.

**Two modes (cell 0):**
- `MODE = "check"`: run cells one by one with Shift+Enter. Safe to use Run All: the full run is skipped. Covers setup, unit tests, data, the longest records and a 16-record smoke run (~25 min).
- `MODE = "full"`: the real 800-record run. Set it, then **Save Version → Save & Run All (Commit)** and close the tab. It runs for up to 12 h (about 6 h per ratio on 2x T4).

In [ ]:
# 0. CONFIG
MODE = "check"        # "check" (interactive) or "full" (Save Version)
RATIOS = ["0.75"]     # for MODE="full". Order: 0.75, 0.9, 0.8, 0.95. One ratio per commit until you know the speed.

In [ ]:
# 1. GPUs (expect 2 x Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Token + team repo + RDKV files from the uploaded zip
import os, sys, glob, zipfile, shutil, subprocess
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def get(*names):
    for n in names:
        try: return sec.get_secret(n)
        except Exception: pass
    return None
os.environ["HF_TOKEN"] = get("HF_TOKEN", "hugging face", "huggingface", "HUGGINGFACE_TOKEN") or ""
print("HF token found:", bool(os.environ["HF_TOKEN"]))

REPO = "/kaggle/working/repo"
if not os.path.exists(REPO + "/harness/run_longbench.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/samruddhichitnis02/298a-kv-cache-compression.git", REPO], check=True)

# overlay: members/manjot/* and the patched harness/run_longbench.py
src = None
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    names = zipfile.ZipFile(z).namelist()
    if "members/manjot/rdkv_press.py" in names and "members/manjot/V4" in names:
        zipfile.ZipFile(z).extractall("/kaggle/working/rdkv_v4"); src = "/kaggle/working/rdkv_v4"; break
if src is None:  # Kaggle may have unzipped it already
    hits = glob.glob("/kaggle/input/**/members/manjot/V4", recursive=True)
    src = hits[0].rsplit("/members/", 1)[0] if hits else None
assert src, "rdkv_kaggle_v4.zip not found: Add Input -> Upload it as a dataset"
shutil.copytree(src + "/members/manjot", REPO + "/members/manjot", dirs_exist_ok=True)
shutil.copy(src + "/harness/run_longbench.py", REPO + "/harness/run_longbench.py")
os.chdir(REPO)
print("repo ready:", REPO, "| manjot files:", os.listdir("members/manjot"))

In [ ]:
# 3. Install (versions from Anees's README: kvpress 0.5.5)
!pip install -q "kvpress==0.5.5" rouge rapidfuzz datasets peft pytest requests
!pip install -q -U "fire>=0.7"   # old fire imports 'pipes', removed in Python 3.13
import sysconfig
if subprocess.run([sys.executable, "-c", "import fire"], capture_output=True).returncode != 0:
    open(os.path.join(sysconfig.get_paths()["purelib"], "pipes.py"), "w").write("from shlex import quote" + chr(10))
!python -c "import torch, transformers, kvpress; print('torch', torch.__version__, '| transformers', transformers.__version__, '| kvpress OK |', torch.cuda.device_count(), 'GPUs')"

In [ ]:
# 4. Unit tests (expect: 9 passed)
if MODE == "check":
    !python -m pytest -q members/manjot/test_rdkv.py

In [ ]:
# 5. Build the 800-record dev set (not in git) and check it matches the baseline's prompts
import json
DATA = "data/processed_longbench/processed_longbench_dev.jsonl"
if not os.path.exists(DATA) or sum(1 for _ in open(DATA)) < 800:
    if os.path.exists(DATA): os.remove(DATA)
    !python members/samruddhi/build_dev_data.py --token $HF_TOKEN
built = {json.loads(l)["record_id"]: json.loads(l)["prompt_tokens_after"] for l in open(DATA)}
base = {json.loads(l)["record_id"]: json.loads(l)["prompt_tokens"] for l in open("results/longbench/Llama-3.1-8B-Instruct_none_0.0.jsonl")}
match = sum(built.get(k) == v for k, v in base.items())
print(f"{len(built)} records built; prompt length identical to the baseline for {match}/{len(base)}")

In [ ]:
# 6. Check mode: (a) the longest records at ratio 0.75, which ran out of memory before v4, then (b) the 16-record smoke run
if MODE == "check":
    !rm -f results/longbench/*_longtest* results/longbench/*_smoke*
    # narrativeqa's first 4 records include the 32,000- and 23,185-token prompts
    !python harness/run_longbench.py --press rdkv --compression-ratio 0.75 --tasks narrativeqa --per-task 4 --dtype float16 --device-map auto --max-gpu-mem 8.5 --out-tag _longtest
    !python harness/run_longbench.py --press rdkv --compression-ratio 0.9 --per-task 1 --dtype float16 --device-map auto --max-gpu-mem 8.5 --out-tag _smoke
    !python members/manjot/compare.py results/longbench/Llama-3.1-8B-Instruct_rdkv_0.9_smoke.jsonl

In [ ]:
# 8. FULL RUN: 800 records, no --max-prompt-tokens (only when MODE = "full")
if MODE == "full":
    for r in RATIOS:
        !python harness/run_longbench.py --press rdkv --compression-ratio {r} --dtype float16 --device-map auto --max-gpu-mem 8.5
        !python members/manjot/compare.py results/longbench/Llama-3.1-8B-Instruct_rdkv_{r}.jsonl

In [ ]:
# 9. Collect outputs (they appear in the version's Output tab)
!mkdir -p /kaggle/working/out && cp results/longbench/*rdkv* results/longbench/*fp16check* /kaggle/working/out/ 2>/dev/null; ls -la /kaggle/working/out